# Learned Stage-Wise Curvature Tuning on CUB-200

This notebook evaluates Learned Stage-Wise Curvature Tuning (SW-CT) on the CUB-200 bird classification dataset using an ImageNet-pretrained ResNet-50.

CUB-200 is a fine-grained classification task containing many visually similar bird categories. This makes it useful for testing whether stage-wise curvature adaptation can improve the representation of subtle visual differences between closely related classes.

The experiments compare four conditions:

1. **Baseline** — the original pretrained ResNet with ReLU activations and a trained downstream classifier.
2. **Single-Parameter Curvature Tuning (S-CT)** — one global beta selected through validation-based exhaustive search.
3. **Learned SW-CT with fixed initialization** — four trainable stage-wise beta parameters initialized at 0.78.
4. **Learned SW-CT with random initialization** — four trainable stage-wise beta parameters independently initialized between 0.70 and 0.99.

Each condition is evaluated using seeds 42, 43, and 44.

Experiments are conducted for both 20 and 30 epochs to examine whether additional optimization improves classification performance or changes the learned stage-wise beta values.

Wall-clock execution time is also recorded to compare computational cost across Baseline, S-CT, and Learned SW-CT.

In [ ]:
!nvidia-smi

In [ ]:
%cd /content

!git clone -b stage-wise-ct \
    https://github.com/Syed1611/curvature-tuning-research.git

%cd /content/curvature-tuning-research/src/curvature-tuning

!git branch --show-current
!git log -1 --oneline

## Environment Setup

The repository dependencies are installed before running the experiments.

After installation, the Colab runtime should be restarted so that the required package versions are loaded correctly.

In [ ]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

In [1]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
import torch

os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/content/curvature-tuning-research/src/curvature-tuning
Working directory: /content/curvature-tuning-research/src/curvature-tuning
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## Arabic Digits Dataset

Before running the experiments, the Arabic Digits data loaders are checked to confirm that the downstream dataset can be loaded correctly.

The same dataset split and random seed protocol are used for Baseline, S-CT, and Learned SW-CT so that the methods are compared under the same conditions.

In [2]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_cub200",
    train_batch_size=32,
    test_batch_size=128,
    seed=42,
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

README.md:   0%|          | 0.00/2.44k [00:00<?, ?B/s]

cub200.py:   0%|          | 0.00/7.90k [00:00<?, ?B/s]

/root/.cache/huggingface/modules/datasets_modules/datasets/randall-lab--cub200/f7aadd51b57da4b2864cf4d2a50611c450decb80865336428f3544a65bed6d70/cub200.py:55: SyntaxWarning: invalid escape sequence '\s'
  images_df = pd.read_csv(image_paths_path, sep='\s+', header=None, names=["image_id", "file_path"])
/root/.cache/huggingface/modules/datasets_modules/datasets/randall-lab--cub200/f7aadd51b57da4b2864cf4d2a50611c450decb80865336428f3544a65bed6d70/cub200.py:56: SyntaxWarning: invalid escape sequence '\s'
  labels_df = pd.read_csv(image_labels_path, sep='\s+', header=None, names=["image_id", "label"])
/root/.cache/huggingface/modules/datasets_modules/datasets/randall-lab--cub200/f7aadd51b57da4b2864cf4d2a50611c450decb80865336428f3544a65bed6d70/cub200.py:57: SyntaxWarning: invalid escape sequence '\s'
  split_df = pd.read_csv(train_test_split_path, sep='\s+', header=None, names=["image_id", "is_training"])


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Train samples: 4796
Validation samples: 1198
Test samples: 5794
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels: tensor([106, 123, 110,  22,  52, 130, 136, 163,  97,  37])


# ResNet-50 Experiments

ResNet-50 is evaluated first because it was used during the initial development of Learned SW-CT.

The experiment compares Baseline, S-CT, fixed-initialization SW-CT, and random-initialization SW-CT.

All methods use the same ImageNet-pretrained backbone, Arabic Digits dataset split, experimental seeds, and downstream training protocol.

## ResNet-50 — 20-Epoch Evaluation

The first evaluation uses 20 training epochs.

This provides the primary reference experiment and maintains compatibility with the earlier Learned SW-CT work.

### Methods

**Baseline**

The pretrained ResNet-50 retains its original ReLU activations. The backbone is frozen and only the downstream classifier is trained.

**S-CT**

The original Single-Parameter Curvature Tuning method searches over a single beta shared throughout the network. Candidate beta values from 0.70 to 1.00 are evaluated using validation performance, and the best candidate is evaluated on the test set.

The recorded S-CT execution time includes the complete beta search rather than only the final selected beta.

**Learned SW-CT — Fixed Initialization**

Four trainable beta parameters are assigned to the four ResNet stages. All four begin at beta = 0.78.

This preserves the initialization used in the original Learned SW-CT experiments.

**Learned SW-CT — Random Initialization**

Four trainable stage-wise beta parameters are initialized independently between 0.70 and 0.99.

The random initialization is generated deterministically from the experiment seed. This allows the experiment to test whether Learned SW-CT depends on the fixed 0.78 starting value while remaining reproducible.

In [3]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-02 23:19:17.811 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_cub200_resnet50_seed42_epochs20.log
2026-10-02 23:19:17.815 | INFO     | __main__:main:78 - Running on cuda
2026-10-02 23:20:44.057 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 23:20:44.138 | INFO     | __main__:main:116 - Number of trainable parameters: 409800
2026-10-02 23:20:44.138 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 23:21:41.506 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.350296, Accuracy: 0.00%
2026-10-02 23:21:41.951 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 5.044851, Accuracy: 3.47%
2026-10-02 23:21:42.417 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 3.600235, Val Accuracy: 21.29%
2026-10-02 23:21:42.419 | INFO     | train:linear_probe:190 - New best validation accuracy: 21.29 at epoch 1
2026-10-02 23:21:42.520 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 3.391333

In [4]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-03 00:04:56.017 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_cub200_resnet50_seed43_epochs20.log
2026-10-03 00:04:56.020 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 00:06:18.413 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 00:06:18.468 | INFO     | __main__:main:116 - Number of trainable parameters: 409800
2026-10-03 00:06:18.468 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 00:07:22.612 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.253399, Accuracy: 0.00%
2026-10-03 00:07:23.029 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 5.014572, Accuracy: 4.12%
2026-10-03 00:07:23.460 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 3.587239, Val Accuracy: 22.37%
2026-10-03 00:07:23.462 | INFO     | train:linear_probe:190 - New best validation accuracy: 22.37 at epoch 1
2026-10-03 00:07:23.553 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 3.284969

In [5]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-03 00:52:09.897 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_cub200_resnet50_seed44_epochs20.log
2026-10-03 00:52:09.901 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 00:53:42.247 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 00:53:42.335 | INFO     | __main__:main:116 - Number of trainable parameters: 409800
2026-10-03 00:53:42.335 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 00:54:41.820 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.343847, Accuracy: 0.00%
2026-10-03 00:54:42.304 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 5.015661, Accuracy: 4.21%
2026-10-03 00:54:42.839 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 3.575829, Val Accuracy: 23.87%
2026-10-03 00:54:42.841 | INFO     | train:linear_probe:190 - New best validation accuracy: 23.87 at epoch 1
2026-10-03 00:54:42.947 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 3.208296

### Learned SW-CT with Fixed Beta Initialization

The original Learned SW-CT configuration is first reproduced using the fixed initialization:

\[[\beta_1,\beta_2,\beta_3,\beta_4]=[0.78,0.78,0.78,0.78].\]

The initial value is only a starting point. During training, the four beta parameters are optimized independently together with the downstream classifier.

These runs provide the reference against which random initialization will be compared.

In [6]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 01:41:57.787 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet50_seed42_epochs20_initfixed0.78_betalr0.01.log
2026-10-03 01:41:57.861 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 01:43:31.306 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 01:43:31.307 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 01:43:31.393 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 01:43:31.394 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 01:43:31.395 | INFO     | __main__:main:439 - Total trainable parameters: 409804
2026-10-03 01:43:33.890 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.252986, Accuracy: 0.00%
2026-10-03 01:44:31.739 | INFO     | train:train_epoch:4

In [7]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 02:18:48.932 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet50_seed43_epochs20_initfixed0.78_betalr0.01.log
2026-10-03 02:18:49.015 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 02:20:18.073 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 02:20:18.074 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 02:20:18.148 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 02:20:18.149 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 02:20:18.150 | INFO     | __main__:main:439 - Total trainable parameters: 409804
2026-10-03 02:20:20.513 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.353955, Accuracy: 0.00%
2026-10-03 02:21:17.159 | INFO     | train:train_epoch:4

In [8]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 02:55:17.112 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet50_seed44_epochs20_initfixed0.78_betalr0.01.log
2026-10-03 02:55:17.225 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 02:56:37.583 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 02:56:37.584 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 02:56:37.686 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 02:56:37.687 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 02:56:37.688 | INFO     | __main__:main:439 - Total trainable parameters: 409804
2026-10-03 02:56:39.948 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.290947, Accuracy: 0.00%
2026-10-03 02:57:36.546 | INFO     | train:train_epoch:4

### Learned SW-CT with Random Beta Initialization

The fixed beta = 0.78 initialization may raise the question of whether Learned SW-CT performs well only because its starting point was informed by the earlier S-CT experiments.

To test this dependency, each of the four stage-wise beta parameters is now initialized independently from a uniform distribution between 0.70 and 0.99.

The initialization is reproducible: the random generator is seeded using the experiment seed. Therefore, each seed always receives the same initial beta vector.

If random initialization produces similar performance to the fixed 0.78 experiment, this provides evidence that the Learned SW-CT result is not dependent on a single hand-selected initialization.

In [9]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 03:30:43.437 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet50_seed42_epochs20_initrandom_betalr0.01.log
2026-10-03 03:30:43.504 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 03:31:46.910 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-03 03:31:46.910 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 03:31:46.976 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-03 03:31:46.977 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 03:31:46.977 | INFO     | __main__:main:439 - Total trainable parameters: 409804
2026-10-03 03:31:48.607 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.220146, Accuracy: 0.00%
2026-10-03 03:32:47.135 | INFO     | train:t

In [10]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 04:04:58.253 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet50_seed43_epochs20_initrandom_betalr0.01.log
2026-10-03 04:04:58.325 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 04:06:01.646 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-03 04:06:01.646 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 04:06:01.713 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-03 04:06:01.714 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 04:06:01.715 | INFO     | __main__:main:439 - Total trainable parameters: 409804
2026-10-03 04:06:03.333 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.384902, Accuracy: 0.00%
2026-10-03 04:07:01.682 | INFO     | train

In [11]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 04:39:17.850 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet50_seed44_epochs20_initrandom_betalr0.01.log
2026-10-03 04:39:17.918 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 04:40:25.095 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-03 04:40:25.096 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 04:40:25.162 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-03 04:40:25.163 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 04:40:25.164 | INFO     | __main__:main:439 - Total trainable parameters: 409804
2026-10-03 04:40:26.863 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.262124, Accuracy: 3.12%
2026-10-03 04:41:25.270 | INFO     | train

# ResNet-50 — 30-Epoch Evaluation

The same experiments are now repeated for 30 epochs.

The purpose of increasing the training budget is to determine whether the 20-epoch results are limited by insufficient optimization.

If a method improves substantially with 30 epochs, this suggests that additional optimization is useful. If performance remains similar, it indicates that the method has largely stabilized within the original 20-epoch training budget.

All other experimental settings remain unchanged.

For random-initialization SW-CT, the same seed produces exactly the same initial four-beta vector in both the 20-epoch and 30-epoch experiments. Therefore, initialization is held constant while the training duration changes.

Execution time is again recorded to measure the computational cost of extending training from 20 to 30 epochs.

In [12]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-03 05:13:50.049 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_cub200_resnet50_seed42_epochs30.log
2026-10-03 05:13:50.052 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 05:14:54.054 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 05:14:54.110 | INFO     | __main__:main:116 - Number of trainable parameters: 409800
2026-10-03 05:14:54.110 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 05:15:34.649 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.350296, Accuracy: 0.00%
2026-10-03 05:15:35.009 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 5.044851, Accuracy: 3.47%
2026-10-03 05:15:35.379 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 3.600235, Val Accuracy: 21.29%
2026-10-03 05:15:35.381 | INFO     | train:linear_probe:190 - New best validation accuracy: 21.29 at epoch 1
2026-10-03 05:15:35.466 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 3.391333

In [13]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-03 05:54:28.698 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_cub200_resnet50_seed43_epochs30.log
2026-10-03 05:54:28.701 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 05:55:37.172 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 05:55:37.227 | INFO     | __main__:main:116 - Number of trainable parameters: 409800
2026-10-03 05:55:37.227 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 05:56:18.351 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.253399, Accuracy: 0.00%
2026-10-03 05:56:18.719 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 5.014572, Accuracy: 4.12%
2026-10-03 05:56:19.115 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 3.587239, Val Accuracy: 22.37%
2026-10-03 05:56:19.116 | INFO     | train:linear_probe:190 - New best validation accuracy: 22.37 at epoch 1
2026-10-03 05:56:19.196 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 3.284969

In [14]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-03 06:34:48.494 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_cub200_resnet50_seed44_epochs30.log
2026-10-03 06:34:48.497 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 06:35:51.755 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 06:35:51.807 | INFO     | __main__:main:116 - Number of trainable parameters: 409800
2026-10-03 06:35:51.807 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 06:36:30.849 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.343847, Accuracy: 0.00%
2026-10-03 06:36:31.200 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 5.015661, Accuracy: 4.21%
2026-10-03 06:36:31.556 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 3.575829, Val Accuracy: 23.87%
2026-10-03 06:36:31.558 | INFO     | train:linear_probe:190 - New best validation accuracy: 23.87 at epoch 1
2026-10-03 06:36:31.628 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 3.208296

In [15]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-03 07:13:24.952 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet50_seed42_epochs30_initfixed0.78_betalr0.01.log
2026-10-03 07:13:25.014 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 07:14:27.952 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 07:14:27.953 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 07:14:28.019 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 07:14:28.020 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 07:14:28.020 | INFO     | __main__:main:439 - Total trainable parameters: 409804
2026-10-03 07:14:29.649 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.252986, Accuracy: 0.00%
2026-10-03 07:15:28.009 | INFO     | train:train_epoch:4

In [3]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-03 09:20:01.593 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet50_seed43_epochs30_initfixed0.78_betalr0.01.log
2026-10-03 09:20:01.669 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 09:21:29.890 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 09:21:29.890 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 09:21:29.997 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 09:21:29.998 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 09:21:29.998 | INFO     | __main__:main:439 - Total trainable parameters: 409804
2026-10-03 09:21:32.537 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.353955, Accuracy: 0.00%
2026-10-03 09:22:28.622 | INFO     | train:train_epoch:4

In [4]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-03 10:14:01.178 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet50_seed44_epochs30_initfixed0.78_betalr0.01.log
2026-10-03 10:14:01.299 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 10:15:25.804 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 10:15:25.805 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 10:15:25.909 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 10:15:25.910 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 10:15:25.910 | INFO     | __main__:main:439 - Total trainable parameters: 409804
2026-10-03 10:15:28.071 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.290947, Accuracy: 0.00%
2026-10-03 10:16:27.267 | INFO     | train:train_epoch:4

In [5]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-03 11:08:08.366 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet50_seed42_epochs30_initrandom_betalr0.01.log
2026-10-03 11:08:08.464 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 11:09:36.386 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-03 11:09:36.387 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 11:09:36.457 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-03 11:09:36.457 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 11:09:36.458 | INFO     | __main__:main:439 - Total trainable parameters: 409804
2026-10-03 11:09:38.234 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.220146, Accuracy: 0.00%
2026-10-03 11:10:37.863 | INFO     | train:t

In [6]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-03 12:02:01.693 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet50_seed43_epochs30_initrandom_betalr0.01.log
2026-10-03 12:02:01.801 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 12:03:29.968 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-03 12:03:29.969 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 12:03:30.036 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-03 12:03:30.037 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 12:03:30.037 | INFO     | __main__:main:439 - Total trainable parameters: 409804
2026-10-03 12:03:32.252 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.384902, Accuracy: 0.00%
2026-10-03 12:04:29.956 | INFO     | train

In [7]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-03 12:56:34.999 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet50_seed44_epochs30_initrandom_betalr0.01.log
2026-10-03 12:56:35.058 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 12:57:58.652 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-03 12:57:58.653 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 12:57:58.723 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-03 12:57:58.724 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 12:57:58.724 | INFO     | __main__:main:439 - Total trainable parameters: 409804
2026-10-03 12:58:00.685 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.262124, Accuracy: 3.12%
2026-10-03 12:58:57.958 | INFO     | train